In [1]:
from pyspark.sql import SparkSession 
spark = SparkSession.builder.appName("Ex. SQL").getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/08 17:06:50 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/08 17:06:51 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [ ]:
#Atividade proposta: 

'''
    01 - Persistir os dados dos arquivos: clientes, vendas, itensVendas, produtos e vendedores no banco de dados do spark
    02 - Criar uma consulta, em SQL, que mostre cada item vendido com as colunas: nome cliente, data venda, produto, vendedor e valor tt item
'''

In [3]:
#Apontando para o banco desp (necessário por causa do docker)

spark.sql("""
CREATE DATABASE desp
LOCATION 'file:/opt/spark/apps/spark-warehouse/desp.db'
""")

DataFrame[]

In [4]:
spark.sql("use desp")

DataFrame[]

In [10]:
#Persistindo os dados

schemaClientes = "ClienteID INT, cliente STRING, Estado STRING, Genero STRING, Status STRING"
schemaItensVendas = "ProdutoID INT, VendaID INT, Quantidade INT, ValorUnitario FLOAT, ValorTotal FLOAT, Desconto FLOAT, TotalComDesconto FLOAT"
schemaProdutos = "ProdutoID INT, Produto STRING, Preco FLOAT"
schemaVendas = "VendasID INT, VendedorID INT, ClienteID INT, Data STRING, Total FLOAT"
schemaVendedores = "VendedorID INT, Vendedor STRING"


clientes = spark.read.parquet("dados/Clientes.parquet", hearder = False, scheema = schemaClientes)
itensVendas =  spark.read.parquet("dados/ItensVendas.parquet", hearder = False, scheema = schemaItensVendas)
produtos = spark.read.parquet("dados/Produtos.parquet", hearder = False, scheema = schemaProdutos)
vendas = spark.read.parquet("dados/Vendas.parquet", hearder = False, scheema = schemaVendas)
vendedores = spark.read.parquet("dados/Vendedores.parquet", hearder = False, scheema = schemaVendedores)

clientes.write.saveAsTable("clientes")
itensVendas.write.saveAsTable("itensVendas")
produtos.write.saveAsTable("produtos")
vendas.write.saveAsTable("vendas")
vendedores.write.saveAsTable("vendedores")

In [33]:
'''
02 - Criar uma consulta, em SQL, que mostre cada item vendido com as colunas: 
nome cliente, data venda, produto, vendedor e valor tt item

'''
spark.sql("select a.Cliente,b.Data,c.Produto,d.Vendedor,e.ValorTotal from clientes a inner join vendas b on a.ClienteID = b.ClienteID inner join itensVendas e on b.VendasID = e.VendasID inner join produtos c on e.ProdutoID = c.ProdutoID inner join vendedores d on b.VendedorID = d.VendedorID ").show()

#spark.sql("select * from clientes").show()

+-----------------+--------+--------------------+----------------+----------+
|          Cliente|    Data|             Produto|        Vendedor|ValorTotal|
+-----------------+--------+--------------------+----------------+----------+
|   Cosme Zambujal|1/1/2019|Bicicleta Altools...|    Armando Lago|   7820.85|
|   Cosme Zambujal|1/1/2019|Bermuda Predactor...|    Armando Lago|     97.75|
|   Cosme Zambujal|1/1/2019|Camiseta Predacto...|    Armando Lago|     135.0|
|Gertrudes Hidalgo|1/1/2020|Luva De Ciclismo ...|   Iberê Lacerda|     150.4|
| Antão Corte-Real|2/1/2020|Capacete Gometws ...|Jéssica Castelão|     155.0|
| Antão Corte-Real|2/1/2020|Bicicleta Gometws...|Jéssica Castelão|    5932.0|
| Antão Corte-Real|2/1/2019|Bicicleta Altools...|  Hélio Liberato|   7820.85|
| Antão Corte-Real|2/1/2019|Bermuda Predactor...|  Hélio Liberato|     97.75|
| Antão Corte-Real|2/1/2019|Bicicleta Gometws...|  Hélio Liberato|    5910.0|
| Antão Corte-Real|3/1/2018|Bicicleta Gometws...|  Hélio Liberat